# **II) Mistral Classification**

In [ ]:
from ollama import Client
import pandas as pd
import numpy as np
import time

In [ ]:
top_k = 20 # Reduces the probability of generating nonsense. A higher value (e.g. 100) will give more diverse answers, while a lower value (e.g. 10) will be more conservative. (Default: 40)
top_p = 0.5 # Works together with top-k. A higher value (e.g., 0.95) will lead to more diverse text, while a lower value (e.g., 0.5) will generate more focused and conservative text. (Default: 0.9)
temp = 0.3 # The temperature of the model. Increasing the temperature will make the model answer more creatively. (Default: 0.8)

system_prompt = """
    Analyze the tweet provided below to determine the stance on monetary policy expressed by the user. Tweets often feature concise language, hashtags, and abbreviations typical of social media communication. Consider the context and the specific wording to classify the stance into one of the following categories: "Very Hawkish", "Hawkish", "Neutral", "Dovish", and "Very Dovish".

    - "Very Hawkish": This category should be selected if the tweet indicates an aggressive stance towards fighting inflation, likely advocating for significant interest rate hikes or other contractionary measures. Look for strong language supporting drastic monetary tightening.
    - "Hawkish": Choose this if the tweet suggests a tendency towards tightening monetary policy to control inflation. It may not indicate extreme measures but shows a clear preference for policy tightening.
    - "Neutral": This applies if the tweet implies a balanced approach, without clear leanings towards tightening or easing monetary policy. It might suggest contentment with maintaining current policy levels or express views that neither strongly support nor oppose policy changes.
    - "Dovish": Select this category if the tweet points towards a preference for easing monetary policy, likely aiming to stimulate economic growth by advocating for lower interest rates or through quantitative easing.
    - "Very Dovish": This should be chosen if the tweet reflects a strong inclination towards stimulating the economy, suggesting substantial measures for easing monetary policy beyond standard adjustments, like a significant push for quantitative easing or deep cuts in interest rates.

    Consider not only the explicit content but also the tone, hashtags, and any economic indicators or events mentioned to understand the user's stance fully. Provide your classification along with a brief explanation of the key factors that influenced your decision.
    
    Remember, your response should only be a category, nothing more. This task is about classification accuracy, not explanation or detail.
    """

user_prompt = """
    Instructions:
    - Analyze the tweet: "{}"
    - Classify into categories: "Very hawkish," "Hawkish," "Neutral," "Dovish," or "Very dovish.".
    - Do not include any explanations, elaborations, or additional information in your response. Your role is solely to classify, not to explain or discuss, return only the category.
    """

def get_ollama(fomc_text):
    # get Ollama's response
    prompt = user_prompt.format(fomc_text)
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt, 'options':{'top_k': top_k, 'temperature':top_p, 'top_p': temp }}
    ]
    response = Client().chat(model='mistral', messages=messages, stream=True)
    result = [part['message']['content'] for part in response]
    result = "".join(result).lower()

    # Try the first part of the term
    keyword_map = {"neutral": 0, "very hawkish": 2, "hawkish": 1, "dovish": -1, "very dovish": -2}
    # Iterate over each key in the keyword_map
    for keyword, value in keyword_map.items():
        if result.strip().startswith(keyword):
            return value

    # Matching inside the text
    # Check for "very hawkish" or "very dovish" then the rest
    if "very hawkish" in result:
        return 2
    elif "very dovish" in result:
        return -2
    elif "hawkish" in result:
        return 1
    elif "dovish" in result:
        return -1
    elif "neutral" in result:
        return 0
    else:
        return None


In [ ]:
df = pd.read_csv("output_FED_class.csv")
print(df)

keyword_map = {"neutral": 0, "very hawkish": -2, "hawkish": -1, "dovish": +1, "very dovish": +2}

chunk_size = 100
n = len(df)

# Loop over the DataFrame in chunks of 100 rows
for start in range(24234, n, chunk_size):
    end = min(start + chunk_size, n)  # Ensure the end index doesn't exceed the DataFrame's length
    for index, row in df.iloc[start:end].iterrows():
        print(index)
        class_ = get_ollama(row["body"])

        # Optionally process and store results
        df.loc[index, "class"] = class_

    df.to_csv("data/stocktwits/output_FED_class.csv", index=False)
    time.sleep(15)

In [ ]:
df_fed = pd.read_csv('data/stocktwits/FED_class.csv')
df_macro = pd.read_csv('data/stocktwits/MACRO_class.csv')

# Combine
df_combined = pd.concat([df_fed, df_macro], ignore_index=True)

df_combined['date'] = pd.to_datetime(df_combined['date'], errors='coerce')
df_combined['nb_likes'] = pd.to_numeric(df_combined['nb_likes'], errors='coerce').fillna(0)
df_combined['nb_reshares'] = pd.to_numeric(df_combined['nb_reshares'], errors='coerce').fillna(0)
df_combined['class'] = pd.to_numeric(df_combined['class'], errors='coerce').fillna(0)

# Define Engagement
# We add + 1 as a base weight so posts with 0 engagement aren't deleted from the math
df_combined['engagement'] = df_combined['nb_likes'] + df_combined['nb_reshares'] + 1

# Weighted Component
df_combined['weighted_class'] = df_combined['class'] * df_combined['engagement']
grouped = df_combined.resample('W', on='date')
weekly_mpe_index = (grouped['weighted_class'].sum() / grouped['engagement'].sum()).reset_index()

weekly_mpe_index.columns = ['date', 'mpe_index']
weekly_mpe_index['date'] = weekly_mpe_index['date'].dt.tz_localize(None)

weekly_mpe_index.to_csv('data/weekly_mpe_index.csv', index=False)